In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### SPARK AND ADLS CONFIGURATION

In [0]:
# Spark configuration for ADLS Gen2 OAuth authentication
# TODO: Move client_id and client_secret to Azure Key Vault for production security
# Currently hardcoded for development purposes only

try:
    spark.conf.set("fs.azure.account.auth.type.spotifydlstorage.dfs.core.windows.net", "OAuth")
    spark.conf.set("fs.azure.account.oauth.provider.type.spotifydlstorage.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set("fs.azure.account.oauth2.client.id.spotifydlstorage.dfs.core.windows.net", "<your-client-id>")
    spark.conf.set("fs.azure.account.oauth2.client.secret.spotifydlstorage.dfs.core.windows.net", "<your-client-secret>")
    spark.conf.set("fs.azure.account.oauth2.client.endpoint.spotifydlstorage.dfs.core.windows.net", "https://login.microsoftonline.com/<your-tenant-id>/oauth2/token")
    print("Spark config set successfully ✓")
except Exception as e:
    print(f"Failed to set Spark config: {e}")
    raise

In [0]:
SILVER_PATH = "abfss://silver@spotifydlstorage.dfs.core.windows.net/spotify_tracks/"
GOLD_PATH = "abfss://gold@spotifydlstorage.dfs.core.windows.net/"

In [0]:
df_silver = spark.read.format("delta")\
                      .load("abfss://silver@spotifydlstorage.dfs.core.windows.net/spotify_tracks/")

In [0]:
df_silver.count()
df_silver.printSchema()

### AGGREGATION

### GENRE AUDIO DNA

In [0]:
df_genre = df_silver.groupBy("track_genre").agg(\
    round(avg("danceability"),3).alias("avg_danceability"),
    round(avg("energy"),3).alias("avg_energy"),
    round(avg("valence"),3).alias("avg_valence"),
    round(avg("tempo"),3).alias("avg_tempo"),
    round(avg("acousticness"),3).alias("avg_acousticness"),
    count("track_id").alias("count_tracks")
).orderBy(desc("count_tracks"))
df_genre.show()

#### GENRE_AUDIO_DNA SAVE INSIDE GOLD CONTAINER

In [0]:
try:
    df_genre.write.format("parquet")\
                  .mode("overwrite")\
                  .save(f"{GOLD_PATH}genre_audio_dna/")
    print("genre_audio_dna written successfully")
except Exception as e:
    print(f"Failed to write genre_audio_dna: {e}")
    raise

### SONG POPULARITY 

#### POPULARITY BY EXPLICIT

In [0]:
df_popularity_explicit = df_silver.groupBy("explicit").agg(
    round(avg("popularity"),3).alias("avg_popularity"),
    count("track_id").alias("track_count")
)
df_popularity_explicit.show()

#### POPULARITY BY MODE

In [0]:
df_popularity_mode = df_silver.groupBy("mode").agg(
    round(avg("popularity"),3).alias("avg_popularity"),
    count("track_id").alias("track_count")
)
df_popularity_mode.show()

#### POPULARITY BY TIME SIGNATURE


In [0]:
df_popularity_time_signature = df_silver.groupBy("time_signature").agg(
    round(avg("popularity"),3).alias("avg_popularity"),
    count("track_id").alias("track_count")
)
df_popularity_time_signature.show()

#### POPULARITY SAVE INSIDE GOLD CONTAINER

In [0]:
popularity_tables = {
    "popularity_by_explicit": df_popularity_explicit,
    "popularity_by_mode": df_popularity_mode,
    "popularity_by_time_signature": df_popularity_time_signature
}

try:
    for name, df in popularity_tables.items():
        df.write.format("parquet")\
                .mode("overwrite")\
                .save(f"{GOLD_PATH}{name}/")
        print(f"Written: {name}")
except Exception as e:
    print(f"Failed to write popularity tables: {e}")
    raise

### TOP ARTIST 

In [0]:
df_explode = df_silver.withColumn("artist", explode(split(col("artists"), ";")))
df_top_artist = df_explode.groupBy("artist").agg(
    round(avg("popularity"),3).alias("avg_popularity"),
    round(avg("danceability"), 3).alias("avg_danceability"),
    count("track_id").alias("count_track")
).filter(col("count_track") >= 3 ).orderBy(desc("avg_popularity"))
df_top_artist.show(20)

#### TOP ARTIST SAVE INSIDE GOLD CONTAINER


In [0]:
try:
    df_top_artist.write.format("parquet")\
                  .mode("overwrite")\
                  .save(f"{GOLD_PATH}top_artist/")
    print("top_artist written successfully")
except Exception as e:
    print(f"Failed to write top_artist: {e}")
    raise

#### MOOD MAP

In [0]:
df_mood = df_silver.withColumn("mood",when((col("energy") >= 0.5) & (col("valence") >= 0.5), "Happy")\
         .when((col("energy") >= 0.5) & (col("valence") < 0.5), "Angry")\
         .when((col("energy") < 0.5) & (col("valence") >= 0.5), "Peaceful")\
         .when((col("energy") < 0.5) & (col("valence") < 0.5), "Sad")\
         .otherwise("Unknown")
         )
df_mood_summary = df_mood.groupBy("mood").agg(
                          count("track_id").alias("count_track"),
                          round(avg("popularity"),3).alias("avg_popularity"),
                          round(avg("danceability"),3).alias("avg_danceability"),
                          round(avg("energy"),3).alias("avg_energy"),
                          round(avg("valence"),3).alias("avg_valence")
).orderBy(desc("avg_popularity"))
df_mood_summary.show()

#### MOOD SUMMARY SAVE INSIDE GOLD CONTAINER

In [0]:
try:
    df_mood_summary.write.format("parquet")\
                  .mode("overwrite")\
                  .save(f"{GOLD_PATH}mood_summary/")
    print("mood_summary written successfully")
except Exception as e:
    print(f"Failed to write mood_summary: {e}")
    raise

### SONG LENGTH VS POPULARITY

In [0]:
df_duration_bucket = df_silver.withColumn("duration_bucket",when(col("duration_min") >= 4, "Long")
                              .when((col("duration_min") >= 2) & (col("duration_min") < 4), "Medium")\
                              .when(col("duration_min") < 2, "Short")\
                              .otherwise("Unknown")   
                    )
df_duration_summary = df_duration_bucket.groupBy("duration_bucket").agg(
                                         count("track_id").alias("count_track"),
                                         round(avg("popularity"),3).alias("avg_popularity")           
).orderBy(desc("avg_popularity"))

df_duration_summary.show()

#### DATA QUALITY ASSERTIONS

In [0]:
try:
    assert df_genre.count() == 114, "Expected 114 genres"
    assert df_mood_summary.count() == 4, "Expected 4 moods"
    assert df_popularity_explicit.count() == 2, "Expected 2 explicit categories"
    assert df_popularity_mode.count() == 2, "Expected 2 modes"
    assert df_popularity_time_signature.count() == 5, "Expected 5 time signatures"
    assert df_top_artist.count() > 0, "Top artist DataFrame is empty"
    assert df_duration_summary.count() == 3, "Expected 3 duration buckets"
    print("All gold assertions passed ✓")
except AssertionError as e:
    print(f"Assertion failed: {e}")
    raise

#### SONG LENGTH VS POPULARITY SAVE INSIDE GOLD CONTAINER

In [0]:
try:
    df_duration_summary.write.format("parquet")\
                  .mode("overwrite")\
                  .save(f"{GOLD_PATH}song_length_vs_popularity/")
    print("song_length_vs_popularity written successfully")
except Exception as e:
    print(f"Failed to write song_length_vs_popularity: {e}")
    raise